In [ ]:
# NAME: Omar Alfaro | NUMBER: 6907

In [ ]:

"""
OKC Analyst Intern Project - NBA shot make probability model

Runs end to end with no other files required:
  training.csv.gz + testing.csv.gz  ->  fits LightGBM  ->  fills submission.csv

Validation approach (see notebook for the full record):
  The test set is an entirely different season from the two in the training set,
  so validation is leave-one-season-out rather than random k-fold. A random split
  would mix shots from the same season into both train and validation and report
  a score better than the model will actually achieve on an unseen season.

Results on that validation:
  constant baseline (predict league make rate)   0.6897
  logistic regression with distance splines      0.6380
  this model (LightGBM, 24 features)             0.6258
  Calibration verified on the diagonal across the full range, so no
  post-hoc calibration is applied.
"""

import numpy as np
import pandas as pd
import lightgbm as lgb

TRAIN_PATH = "training.csv.gz"
TEST_PATH = "testing.csv.gz"
SUBMISSION_PATH = "submission.csv"

# Rim center, from the court diagram in the project brief.
HX, HY = -41.75, 0.0

# Mean early-stopping point across the two season folds.
N_TREES = 162

PARAMS = dict(
    objective="binary",          # optimize log-loss directly, the scored metric
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=200,       # no leaf built on a handful of shots
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbose=-1,
)


def parse_approach(df):
    """closestdefapproach is '{a,b,c,d}': the closest defender's distance at
    1.0s, 0.75s, 0.5s and 0.25s before release. Returns them as four columns."""
    a = (df.closestdefapproach.str.strip("{}")
         .str.split(",", expand=True).astype(float))
    a.columns = [f"a{i}" for i in range(a.shape[1])]
    return a


def build_features(df, shot_counts):
    """Identical treatment for train and test. shot_counts is passed in rather
    than computed here so the test set can never contribute to it."""
    out = pd.DataFrame(index=df.index)

    # --- core shot context ---
    # Distance capped at 35 ft: past that the sample thins to a few hundred
    # shots and the observed make rate is mostly noise. shottype == 'heave'
    # still flags those shots as different.
    out["distance"] = df.distance.clip(upper=35)
    out["shotclock"] = df.shotclock
    # Defender distance has the same shape as the distance curve: 2 ft vs 4 ft
    # matters enormously, 15 ft vs 17 ft barely at all. log1p compresses the
    # far end so the model spends its splits where the differences are real.
    out["defdist"] = np.log1p(df.closestdefdist.clip(upper=20))
    out["dribbles"] = df.dribblesbefore.clip(upper=15)
    out["speed"] = df.shooterspeed
    out["contesters"] = df.num_contesters
    out["three"] = df.three.astype(int)
    # The closest defender is not always a contester. On ~10k training shots a
    # defender is nearby but not challenging the shot, which is a real
    # situation rather than a data problem.
    out["closest_not_contesting"] = ((df.num_contesters > 0) &
                                     (df.closestdef_id != df.contester1_id)).astype(int)
    out["shottype"] = df.shottype
    out["gamestate"] = df.gamestate

    # --- location ---
    # distance alone collapses every point on a ring around the hoop into one
    # number, so a corner three and a straightaway three look identical.
    out["abs_y"] = (df.locationy - HY).abs()
    out["angle"] = np.degrees(np.arctan2((df.locationy - HY).abs(),
                                         (df.locationx - HX).abs()))
    out["corner3"] = (df.three & ((df.locationy - HY).abs() > 20)).astype(int)

    # --- defender geometry: where the contesters are ---
    # num_contesters counts bodies but says nothing about spacing. Two
    # defenders both within 4 ft is a different shot from one at 3 ft and one
    # at 15 ft. Missing values here are structural (no second contester
    # existed) and LightGBM routes them down their own branch.
    d = df[["distcont1", "distcont2", "distcont3", "distcont4"]]
    out["cont_min"] = d.min(axis=1)
    out["cont_mean"] = d.mean(axis=1)
    out["cont_2nd"] = df.distcont2
    out["n_within_4"] = (d < 4).sum(axis=1)
    out["n_within_6"] = (d < 6).sum(axis=1)

    # --- defender geometry: closing motion over the last second ---
    # closestdefdist is a single frozen snapshot at release. These capture what
    # the defender was doing on the way there, which partly reveals the
    # defense's own read of the shot.
    a = parse_approach(df)
    out["def_start"] = a.a0                          # distance 1.0s out
    out["def_delta"] = a.a3 - a.a0                   # ground covered over 1.0s
    out["def_late"] = a.a3 - a.a2                    # ground covered in last 0.25s
    out["def_accel"] = (a.a3 - a.a2) - (a.a1 - a.a0)  # late lunge vs steady close
    out["close_rate_vs_speed"] = (a.a0 - a.a3) / (df.shooterspeed + 1)

    # --- shooter ---
    # Shot volume, not make rate. A count contains no outcome information, so
    # it cannot leak, and it captures whether this is a player the offense
    # routinely shoots with. Unseen players in the test season get 0, which
    # honestly describes them as low-usage unknowns.
    out["shooter_vol"] = df.shooter_id.map(shot_counts).fillna(0)
    return out


def main():
    print("loading data")
    train = pd.read_csv(TRAIN_PATH, na_values=["NA"])
    test = pd.read_csv(TEST_PATH, na_values=["NA"])

    # Computed from training seasons only. Using the test season's own counts
    # would be peeking at data unavailable in a real forecasting setting.
    shot_counts = train.shooter_id.value_counts()

    print("building features")
    X = build_features(train, shot_counts)
    X_test = build_features(test, shot_counts)
    y = train.outcome.astype(int)

    # Shared category ordering, or the two files encode differently in silence.
    for c in ["shottype", "gamestate"]:
        X[c] = X[c].astype("category")
        X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)
    X_test = X_test[X.columns]
    print(f"  {X.shape[1]} features | train {X.shape[0]:,} | test {X_test.shape[0]:,}")

    print(f"fitting LightGBM on both seasons ({N_TREES} trees)")
    model = lgb.LGBMClassifier(**PARAMS, n_estimators=N_TREES)
    model.fit(X, y)

    preds = pd.DataFrame({"shot_id": test.shot_id,
                          "make_prob": model.predict_proba(X_test)[:, 1]})

    # Merge onto the template's shot_id column so row order is preserved
    # exactly and no column is added, removed, or renamed.
    sub = pd.read_csv(SUBMISSION_PATH)
    sub = sub[["shot_id"]].merge(preds, on="shot_id", how="left")

    assert len(sub) == len(test), f"expected {len(test)} rows, got {len(sub)}"
    assert sub.make_prob.notna().all(), "some shots have no prediction"
    assert sub.make_prob.between(0, 1).all(), "predictions outside [0, 1]"
    assert list(sub.columns) == ["shot_id", "make_prob"], "column mismatch"

    sub.to_csv(SUBMISSION_PATH, index=False)

    print(f"\nwrote {SUBMISSION_PATH}")
    print(f"  rows          {len(sub):,}")
    print(f"  mean          {sub.make_prob.mean():.4f}  (train actual {y.mean():.4f})")
    print(f"  range         {sub.make_prob.min():.4f} to {sub.make_prob.max():.4f}")

    imp = pd.Series(model.booster_.feature_importance("gain"), index=X.columns)
    imp = (imp / imp.sum() * 100).sort_values(ascending=False)
    print("\ntop features by gain (%):")
    print(imp.head(8).round(1).to_string())


if __name__ == "__main__":
    main()

loading data
building features
  24 features | train 425,719 | test 213,977
fitting LightGBM on both seasons (162 trees)

wrote submission.csv
  rows          213,977
  mean          0.4547  (train actual 0.4581)
  range         0.0465 to 0.9918

top features by gain (%):
distance      29.2
shottype      23.5
cont_min       8.1
abs_y          5.5
contesters     4.7
defdist        3.6
speed          3.3
shotclock      3.1
